In [1]:
!find /kaggle/input -maxdepth 4 -type d -name lake

/kaggle/input/notebooks/nawresbm2/05-spark-lakehouse/lake


In [2]:
!pip install -q pyspark
import glob
from functools import reduce
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]").config("spark.driver.memory", "16g")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.sources.partitionColumnTypeInference.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

lakes = sorted(glob.glob("/kaggle/input/**/lake", recursive=True))
print("Lac trouvé :", lakes)
LAKE = lakes[0]
silver = spark.read.parquet(f"{LAKE}/silver/alert_flow")

def bronze(name):
    paths = sorted(glob.glob(f"{LAKE}/bronze/{name}/file=*"))
    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True),
                  [spark.read.parquet(p).drop("date") for p in paths])

print(silver.groupBy("direction").count().collect())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 21:44:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Lac trouvé : ['/kaggle/input/notebooks/nawresbm2/05-spark-lakehouse/lake']


[Row(direction='entrant', count=2582811), Row(direction='interne', count=960453), Row(direction='sortant', count=95850)]


In [3]:
out = silver.where("direction = 'sortant' AND only_noise = 0").cache()
inb = silver.where("direction = 'entrant' AND only_noise = 0")
n_out = out.count()
print(f"Flux sortants (hors bruit) : {n_out:,} | taux de positifs v1 : {out.agg(F.avg('label_v1')).first()[0]:.3f}")

print("\n1) Machine interne à l'origine :")
out.groupBy("src_ip").count().orderBy(F.desc("count")).show(5)

print("2) Répartition par fichier (période) :")
out.groupBy("source_file").agg(F.count("*").alias("flux"), F.round(F.avg("label_v1"), 3).alias("taux_v1"),
                               F.countDistinct("dest_ip").alias("ip_dest")).orderBy("source_file").show()

print("3) Ports et protocoles de destination :")
out.groupBy("dest_port", "proto", "app_proto").count().orderBy(F.desc("count")).show(8)

print("4) IP de destination les plus contactées :")
out.groupBy("dest_ip").agg(F.count("*").alias("flux"), F.round(F.avg("label_v1"), 3).alias("taux_v1"),
                           F.min("ts").alias("premier"), F.max("ts").alias("dernier")).orderBy(F.desc("flux")).show(10, truncate=False)

print("5) Signature la plus sévère par flux :")
out.groupBy("sig", "sig_category", "sig_severity").count().orderBy(F.desc("count")).show(12, truncate=80)

print("6) Toutes les signatures levées sur ces flux :")
al = bronze("alert").join(F.broadcast(out.select("flow_id")), "flow_id")
al.groupBy("signature").count().orderBy(F.desc("count")).show(12, truncate=80)

print("7) Les destinations sont-elles des IP qui attaquent aussi le serveur (vues en entrant) ?")
dests = out.select(F.col("dest_ip").alias("ip")).distinct()
seen_in = inb.select(F.col("src_ip").alias("ip")).distinct()
attackers = inb.where("label_v1 = 1").select(F.col("src_ip").alias("ip")).distinct()
nd = dests.count()
n_seen, n_att = dests.join(seen_in, "ip").count(), dests.join(attackers, "ip").count()
print(f"   IP de destination distinctes : {nd:,} | déjà vues en entrant : {n_seen:,} ({n_seen / nd:.1%}) | "
      f"dont attaquantes (positif en entrant) : {n_att:,} ({n_att / nd:.1%})")

print("\n8) Évolution mensuelle (volume et octets) :")
(out.groupBy(F.date_format("ts", "yyyy-MM").alias("mois"))
    .agg(F.count("*").alias("flux"), F.countDistinct("dest_ip").alias("ip_dest"),
         F.round(F.avg("bytes_toserver"), 0).alias("octets_envoyes_moy"),
         F.round(F.avg("bytes_toclient"), 0).alias("octets_recus_moy"))
    .orderBy("mois").show(15))

Flux sortants (hors bruit) : 46,279 | taux de positifs v1 : 0.973

1) Machine interne à l'origine :
+------------+-----+
|      src_ip|count|
+------------+-----+
|192.168.33.8|45053|
|192.168.33.9| 1226|
+------------+-----+

2) Répartition par fichier (période) :
+-----------+-----+-------+-------+
|source_file| flux|taux_v1|ip_dest|
+-----------+-----+-------+-------+
|        001|  172|  0.012|      7|
|        002|  115|    0.0|      8|
|        003|  101|    0.0|      6|
|        004| 4247|  0.973|     10|
|        005|14484|  0.981|      7|
|        006|14734|  0.975|      8|
|        007|12415|  0.992|      7|
|        008|   11|    1.0|      1|
+-----------+-----+-------+-------+

3) Ports et protocoles de destination :
+---------+-----+---------+-----+
|dest_port|proto|app_proto|count|
+---------+-----+---------+-----+
|       53|  UDP|      dns|45049|
|       80|  TCP|     http| 1226|
|      443|  TCP|      tls|    3|
|       22|  TCP|     NULL|    1|
+---------+-----+------

+--------------------------------------------------------------------------------+-----+
|                                                                       signature|count|
+--------------------------------------------------------------------------------+-----+
|          ET POLICY External IP Lookup Domain (myip .opendns .com in DNS lookup)|93460|
|ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management| 3662|
|                                    ET HUNTING Telegram API Domain in DNS Lookup|  124|
|                                              ET POLICY curl User-Agent Outbound|   10|
|                              ET POLICY External IP Lookup Domain (ifconfig .me)|    8|
|                                       SURICATA STREAM excessive retransmissions|    3|
|         ET HUNTING Observed Telegram API Domain (api .telegram .org in TLS SNI)|    3|
|                                      GPL ATTACK_RESPONSE id check returned root|    2|
|                    

   IP de destination distinctes : 15 | déjà vues en entrant : 0 (0.0%) | dont attaquantes (positif en entrant) : 0 (0.0%)

8) Évolution mensuelle (volume et octets) :
+-------+----+-------+------------------+----------------+
|   mois|flux|ip_dest|octets_envoyes_moy|octets_recus_moy|
+-------+----+-------+------------------+----------------+
|2024-03|  55|      7|           79902.0|       4325266.0|
|2024-04|  74|      5|           96780.0|       5666491.0|
|2024-05|  76|      5|           97532.0|       5252090.0|
|2024-06|  72|      8|           98220.0|       5756639.0|
|2024-07|  73|      6|          121895.0|       6448841.0|
|2024-08|  70|      6|          103579.0|       6145176.0|
|2024-09|7079|     10|            1396.0|         63861.0|
|2024-10|9154|      7|             734.0|         26958.0|
|2024-11|9106|      8|             261.0|          1021.0|
|2024-12|8650|      7|            1464.0|         85101.0|
|2025-01|8826|      7|            1157.0|         48859.0|
|2025-0

In [4]:
from pyspark.sql import Window

MYIP = "ET POLICY External IP Lookup Domain (myip .opendns .com in DNS lookup)"
APT = "ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management"
myip = out.where(F.col("sig") == MYIP)
first = myip.agg(F.min("ts")).first()[0]
print("Première requête myip.opendns.com :", first)

# Périodicité de la tâche (intervalle entre deux requêtes, en secondes)
g = myip.select("ts").withColumn("gap", F.col("ts").cast("double") - F.lag(F.col("ts").cast("double")).over(Window.orderBy("ts")))
print("Intervalle entre requêtes (p10 / médiane / p90, en s) :",
      g.select(F.expr("percentile_approx(gap, array(0.1, 0.5, 0.9))")).first()[0])

print("\nA) Événements sortants rares (hors myip et APT), par ordre chronologique :")
out.where(~F.col("sig").isin(MYIP, APT)).select("ts", "src_ip", "dest_ip", "dest_port", "sig") \
   .orderBy("ts").show(50, truncate=70)

allal = bronze("alert")
print("B) Toutes les alertes ATTACK_RESPONSE (toutes directions) :")
allal.where(F.col("signature").contains("ATTACK_RESPONSE")) \
     .select("ts", "src_ip", "src_port", "dest_ip", "dest_port", "app_proto", "signature").orderBy("ts").show(20, truncate=60)

win = (F.col("ts") >= F.lit(first) - F.expr("INTERVAL 72 HOURS")) & (F.col("ts") <= F.lit(first))
print("C) Alertes sévères (sévérité <= 2) impliquant le serveur applicatif 192.168.33.8, 72 h avant :")
allal.where(win & ((F.col("dest_ip") == "192.168.33.8") | (F.col("src_ip") == "192.168.33.8")) & (F.col("severity") <= 2)) \
     .groupBy("signature").agg(F.count("*").alias("n"), F.min("ts").alias("premiere"), F.max("ts").alias("derniere")) \
     .orderBy(F.desc("n")).show(15, truncate=70)

print("D) Alertes de sévérité 1 vers le proxy 192.168.33.9, 72 h avant :")
allal.where(win & (F.col("dest_ip") == "192.168.33.9") & (F.col("severity") == 1)) \
     .groupBy("signature").agg(F.count("*").alias("n"), F.countDistinct("src_ip").alias("ip_src"), F.max("ts").alias("derniere")) \
     .orderBy(F.desc("n")).show(15, truncate=70)

Première requête myip.opendns.com : 2024-09-06 15:35:01.638119
Intervalle entre requêtes (p10 / médiane / p90, en s) : [299.37827491760254, 300.0117199420929, 300.3734049797058]

A) Événements sortants rares (hors myip et APT), par ordre chronologique :
+--------------------------+------------+---------------+---------+----------------------------------------------------------------------+
|                        ts|      src_ip|        dest_ip|dest_port|                                                                   sig|
+--------------------------+------------+---------------+---------+----------------------------------------------------------------------+
|2024-03-12 09:51:42.669688|192.168.33.9|  18.155.68.104|       80|                                    ET POLICY curl User-Agent Outbound|
|2024-03-12 09:52:14.457377|192.168.33.9|   18.155.68.70|       80|                                    ET POLICY curl User-Agent Outbound|
| 2024-09-06 16:35:43.50178|192.168.33.9|149.154.16

B) Toutes les alertes ATTACK_RESPONSE (toutes directions) :


+--------------------------+--------------+--------+------------+---------+---------+------------------------------------------------------------+
|                        ts|        src_ip|src_port|     dest_ip|dest_port|app_proto|                                                   signature|
+--------------------------+--------------+--------+------------+---------+---------+------------------------------------------------------------+
|2024-03-12 09:51:42.701448| 18.155.68.104|      80|192.168.33.9|    39916|     http|                  GPL ATTACK_RESPONSE id check returned root|
| 2024-03-12 09:52:14.48726|  18.155.68.70|      80|192.168.33.9|    47392|     http|                  GPL ATTACK_RESPONSE id check returned root|
|2024-04-21 18:20:08.751953| 152.42.212.39|   49639|192.168.33.9|       80|     http|ET ATTACK_RESPONSE Possible WebShell Upload Attempt via D...|
|2024-04-21 18:20:08.751998|  192.168.33.9|   49639|192.168.33.8|       80|     http|ET ATTACK_RESPONSE Possible WebSh

+----------------------------------------------------------------------+---+--------------------------+--------------------------+
|                                                             signature|  n|                  premiere|                  derniere|
+----------------------------------------------------------------------+---+--------------------------+--------------------------+
|                 ET POLICY Http Client Body contains pwd= in cleartext|118|2024-09-03 16:00:43.396571|  2024-09-06 13:17:28.8152|
|                                   ET POLICY Cleartext WordPress Login|113|2024-09-03 16:00:43.396571|  2024-09-06 13:17:28.8152|
|       ET SCAN Laravel Debug Mode Information Disclosure Probe Inbound| 17|2024-09-03 20:00:42.657179|2024-09-05 23:54:37.933784|
|ET EXPLOIT Apache HTTP Server - Path Traversal Attempt (CVE-2021-42...|  4|2024-09-04 06:46:37.189568|2024-09-06 09:28:58.822615|
|                ET HUNTING Suspicious PHP Code in HTTP POST (Outbound)|  4|2024-09

+----------------------------------------------------------------------+---+------+--------------------------+
|                                                             signature|  n|ip_src|                  derniere|
+----------------------------------------------------------------------+---+------+--------------------------+
|                                   ET POLICY Cleartext WordPress Login|113|    26|2024-09-06 13:17:28.815162|
|                                   ET WEB_SERVER PHP tags in HTTP POST| 72|     9|2024-09-06 11:44:37.619498|
|ET EXPLOIT Telesquare SDT-CW3B1 1.1.0 - OS Command Injection (CVE-2...| 28|     1|2024-09-04 10:37:51.373846|
| ET WEB_SERVER /bin/sh In URI Possible Shell Command Execution Attempt|  8|     4|2024-09-06 09:28:58.822599|
|ET EXPLOIT file_put_contents php base64 encoded Remote Code Executi...|  6|     4|2024-09-06 05:49:35.600696|
|              ET WEB_SERVER allow_url_include PHP config option in uri|  4|     4|2024-09-06 09:28:58.921807|
|

In [5]:
day = (F.col("ts") >= F.lit("2024-09-03")) & (F.col("ts") <= F.lit(first))

print("E) L'alerte 'ET COMPROMISED' juste avant 15:35 :")
allal.where(F.col("signature").contains("COMPROMISED") & win) \
     .select("ts", "src_ip", "src_port", "dest_ip", "dest_port", "proto", "app_proto", "signature").orderBy("ts").show(truncate=60)

print("F) Alertes ATTACK_RESPONSE par mois (toute la période) :")
allal.where(F.col("signature").contains("ATTACK_RESPONSE")) \
     .groupBy(F.date_format("ts", "yyyy-MM").alias("mois"), "signature").count().orderBy("mois").show(30, truncate=70)

h = bronze("http")
wp = h.where(day & F.col("url").contains("wp-login.php") & (F.col("http_method") == "POST"))
print("G) Connexions WordPress (POST /wp-login.php) du 3 au 6 septembre, par code HTTP :")
wp.groupBy("status").count().orderBy(F.desc("count")).show()
print("   Connexions probablement réussies (302 = redirection après authentification) :")
wp.where(F.col("status") == 302).select("ts", "src_ip", "dest_ip", "url", "http_user_agent").orderBy("ts").show(20, truncate=60)

print("H) Accès à l'administration WordPress (/wp-admin) le 6 septembre avant 15:35 :")
(h.where((F.col("ts") >= F.lit("2024-09-06")) & (F.col("ts") <= F.lit(first)) & F.col("url").contains("wp-admin"))
  .groupBy("src_ip", "http_method", "status")
  .agg(F.count("*").alias("n"), F.min("ts").alias("debut"), F.max("ts").alias("fin"), F.first("url").alias("exemple_url"))
  .orderBy(F.desc("n")).show(20, truncate=60))

E) L'alerte 'ET COMPROMISED' juste avant 15:35 :


+--------------------------+---------------+--------+------------+---------+-----+---------+------------------------------------------------------------+
|                        ts|         src_ip|src_port|     dest_ip|dest_port|proto|app_proto|                                                   signature|
+--------------------------+---------------+--------+------------+---------+-----+---------+------------------------------------------------------------+
|2024-09-03 16:39:00.667641|185.191.126.213|   56656|192.168.33.9|       80|  TCP|     NULL|ET COMPROMISED Known Compromised or Hostile Host Traffic ...|
|2024-09-03 17:38:28.721433|185.191.126.213|   53831|192.168.33.9|       80|  TCP|     NULL|ET COMPROMISED Known Compromised or Hostile Host Traffic ...|
|2024-09-03 18:19:02.165102|185.191.126.213|   33229|192.168.33.9|       80|  TCP|     NULL|ET COMPROMISED Known Compromised or Hostile Host Traffic ...|
|2024-09-03 19:44:51.428252|185.191.126.213|   48789|192.168.33.9|       80|

+-------+----------------------------------------------------------------------+-----+
|   mois|                                                             signature|count|
+-------+----------------------------------------------------------------------+-----+
|2024-03|                            GPL ATTACK_RESPONSE id check returned root|    2|
|2024-04|ET ATTACK_RESPONSE Possible WebShell Upload Attempt via Directory T...|   16|
|2024-06|ET ATTACK_RESPONSE Possible WebShell Upload Attempt via Directory T...|    2|
|2024-07|ET ATTACK_RESPONSE Possible WebShell Upload Attempt via Directory T...|   44|
|2024-11|ET ATTACK_RESPONSE Possible WebShell Upload Attempt via Directory T...|    2|
+-------+----------------------------------------------------------------------+-----+



G) Connexions WordPress (POST /wp-login.php) du 3 au 6 septembre, par code HTTP :


+------+-----+
|status|count|
+------+-----+
|   404|  277|
|  NULL|   15|
+------+-----+

   Connexions probablement réussies (302 = redirection après authentification) :


+---+------+-------+---+---------------+
| ts|src_ip|dest_ip|url|http_user_agent|
+---+------+-------+---+---------------+
+---+------+-------+---+---------------+

H) Accès à l'administration WordPress (/wp-admin) le 6 septembre avant 15:35 :


+---------------+-----------+------+---+--------------------------+--------------------------+------------------------------------------------------------+
|         src_ip|http_method|status|  n|                     debut|                       fin|                                                 exemple_url|
+---------------+-----------+------+---+--------------------------+--------------------------+------------------------------------------------------------+
|   192.168.33.9|        GET|   404|117|2024-09-06 00:15:05.528898|2024-09-06 12:44:40.967508|                                                  /wp-admin/|
| 131.153.227.11|        GET|   404| 81|2024-09-06 05:45:37.121786|2024-09-06 05:49:29.381882|                                          /wp-admin/bala.php|
| 43.229.114.110|        GET|   404| 22|2024-09-06 08:20:21.867983|2024-09-06 11:56:50.068756|/wp-admin/admin-ajax.php?action=handle_downloads&alg_wc_p...|
| 172.104.184.11|        GET|   404| 11|2024-09-06 02:31:27.2578